# Cosmos 3 vs. Industrial AI — Data Gap Analysis

**Google Colab notebook · Dataset research track · 2026-07-23**

Notebook này trực quan hóa phần nào của data curriculum Cosmos 3 có thể tái sử dụng cho Industrial AI và phần nào bắt buộc phải thu thập trong nhà máy.

> Các điểm coverage 0–5 là **analytical rubric phục vụ ra quyết định**, không phải benchmark accuracy của model.

## Cách chạy

1. Mở notebook bằng Google Colab.
2. Chọn **Runtime → Run all**. GPU không bắt buộc.
3. Sau khi chạy xong, tải `cosmos3_industrial_gap_analysis_artifacts.zip` ở panel Files.

Notebook nhúng sẵn dữ liệu phân tích nên không cần clone repository hay đăng nhập Hugging Face.

In [ ]:
%pip install -q -U pandas matplotlib seaborn plotly ipywidgets

import io, json, math, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid', context='notebook')
PLOTLY_CONFIG = {'displaylogo': False, 'responsive': True}
OUT = Path('/content/cosmos3_industrial_gap_analysis')
OUT.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUT)

## Rubric và phạm vi

### Coverage score

| Score | Ý nghĩa |
|---:|---|
| 0 | Không có bằng chứng coverage |
| 1 | Rất yếu / chỉ có proxy gián tiếp |
| 2 | Có một phần nhưng chưa đủ cho nhà máy |
| 3 | Nền tảng hữu ích, cần adaptation đáng kể |
| 4 | Coverage mạnh, cần dữ liệu target-domain |
| 5 | Coverage rất mạnh và trực tiếp |

Phạm vi Industrial AI giả định gồm inspection, machine condition, manual/robot assembly, intralogistics, worker safety, process control và predictive maintenance.

In [ ]:
gap_csv = '''gap_id,category,gap,cosmos_coverage,industrial_requirement,severity,priority,closure_evidence
G01,Machine state,PLC SCADA and machine telemetry,1.0,5.0,5,P0,Real synchronized historian export
G02,Action and causality,Explicit control actions and interventions,2.0,5.0,5,P0,Command log aligned to pre-state and post-state
G03,Outcome,Outcome and business labels,1.5,5.0,5,P0,Traceable quality safety downtime and rework outcomes
G04,Lifecycle,Long-horizon degradation,1.0,5.0,5,P0,Leakage-safe lifecycle and maintenance histories
G05,Domain,Real factory domain anchor,1.5,5.0,5,P0,Held-out real target-factory episodes
G06,Labels,Factory fault taxonomy,2.0,5.0,5,P0,SME-approved taxonomy with confirmed cases
G07,Synchronization,Sensor video audio synchronization,2.0,5.0,5,P0,Measured synchronization and clock-error audit
G08,Diversity,Product and process diversity,2.0,5.0,4,P1,Coverage matrix by machine product shift and site
G09,Action and causality,Counterfactual and causal supervision,2.5,5.0,4,P1,Controlled or matched intervention pairs
G10,Inspection,Fine-grained inspection labels,3.0,5.0,4,P1,Metrology-linked masks measurements and disposition
G11,Audio and vibration,Industrial audio and vibration,2.5,5.0,4,P1,Normal and fault recordings under varied load
G12,Procedure,Manual procedures and assembly state,3.0,5.0,4,P1,SOP-linked multi-view episodes
G13,Data quality,OT data quality and sensor failure,1.0,5.0,4,P1,Dropout drift saturation and calibration flags
G14,Distribution,Normal-operation coverage,1.5,5.0,4,P1,Representative normal baseline distribution
G15,Validation,Site and machine leakage control,1.5,5.0,5,P0,Immutable split manifest by site machine batch operation
G16,Geometry,Calibration and metric geometry,4.0,5.0,3,P1,Real calibration and reprojection audit
G17,Governance,Human privacy and labor sensitivity,2.0,5.0,5,P0,Approved privacy redaction and retention protocol
G18,Governance,OT security and proprietary process protection,0.5,5.0,5,P0,Security-approved access and export plan
G19,Validation,Sim-to-real admission,2.5,5.0,5,P0,Real-only versus real-plus-synthetic ablation
G20,Deployment,Deployment-rate data,2.0,5.0,4,P1,Replay benchmark under production timing constraints
G21,Uncertainty,Uncertainty and abstention labels,2.5,5.0,4,P1,Human-adjudicated ambiguous and OOD calibration set
G22,Safety,Rare catastrophic events,3.0,5.0,4,P2,Hazard analysis followed by controlled simulation'''

gaps = pd.read_csv(io.StringIO(gap_csv))
gaps['coverage_gap'] = gaps['industrial_requirement'] - gaps['cosmos_coverage']
gaps['priority_rank'] = gaps['priority'].map({'P0': 0, 'P1': 1, 'P2': 2})

assert len(gaps) == 22
assert gaps['gap_id'].is_unique
assert gaps['severity'].between(1, 5).all()
assert gaps['cosmos_coverage'].between(0, 5).all()
assert (gaps['coverage_gap'] >= 0).all()

display(Markdown(
    f"**Validated:** {len(gaps)} gaps · "
    f"P0: {(gaps.priority == 'P0').sum()} · "
    f"P1: {(gaps.priority == 'P1').sum()} · "
    f"P2: {(gaps.priority == 'P2').sum()}"
))
display(gaps.head())

## 1. Capability coverage: Cosmos mạnh ở đâu, nhà máy còn thiếu gì?

Biểu đồ dumbbell dùng chung thang 0–5. Khoảng cách càng lớn thì lượng dữ liệu factory-native cần bổ sung càng nhiều.

In [ ]:
capability_rows = [
    ('RGB image/video', 5.0, 5.0, 'Reuse'),
    ('Spatial grounding and geometry', 4.5, 5.0, 'Adapt'),
    ('Physical interaction dynamics', 4.0, 5.0, 'Adapt'),
    ('Robot manipulation actions', 4.0, 4.5, 'Adapt'),
    ('Temporal event understanding', 4.0, 5.0, 'Adapt'),
    ('OCR and document reasoning', 4.0, 4.5, 'Adapt'),
    ('Audio representation', 3.0, 4.5, 'Collect'),
    ('Fine-grained visual inspection', 3.0, 5.0, 'Collect'),
    ('Factory control actions', 2.0, 5.0, 'Collect'),
    ('Uncertainty and OOD', 2.0, 5.0, 'Collect'),
    ('PLC/SCADA telemetry', 1.0, 5.0, 'Collect'),
    ('Real factory coverage', 1.5, 5.0, 'Collect'),
    ('Quality and business outcomes', 1.0, 5.0, 'Collect'),
    ('Long-horizon degradation', 1.0, 5.0, 'Collect'),
    ('Maintenance history and RUL', 0.5, 5.0, 'Collect'),
]
cap = pd.DataFrame(capability_rows, columns=['capability', 'cosmos_coverage', 'industrial_need', 'decision'])
cap['gap'] = cap['industrial_need'] - cap['cosmos_coverage']
cap = cap.sort_values('gap')

fig = go.Figure()
for _, row in cap.iterrows():
    fig.add_trace(go.Scatter(
        x=[row.cosmos_coverage, row.industrial_need],
        y=[row.capability, row.capability],
        mode='lines', line=dict(color='#AAB2BD', width=4),
        hoverinfo='skip', showlegend=False
    ))
fig.add_trace(go.Scatter(
    x=cap.cosmos_coverage, y=cap.capability, mode='markers', name='Cosmos 3 coverage',
    marker=dict(size=11, color='#4C78A8'),
    customdata=cap[['gap', 'decision']],
    hovertemplate='%{y}<br>Coverage: %{x:.1f}/5<br>Gap: %{customdata[0]:.1f}<br>Decision: %{customdata[1]}<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=cap.industrial_need, y=cap.capability, mode='markers', name='Industrial requirement',
    marker=dict(size=11, color='#F58518'),
    hovertemplate='%{y}<br>Required: %{x:.1f}/5<extra></extra>'
))
fig.update_layout(
    title='Capability coverage gap', xaxis_title='Coverage maturity (0–5)',
    height=620, margin=dict(l=240, r=30, t=70, b=50),
    xaxis=dict(range=[0, 5.2], dtick=1), legend=dict(orientation='h', y=1.08)
)
fig.show(config=PLOTLY_CONFIG)
fig.write_html(OUT / '01_capability_gap.html', include_plotlyjs='cdn')

## 2. Gap priority và severity

- **P0:** phải xử lý trước khi train hoặc tuyên bố factory readiness.
- **P1:** cần trước pilot.
- **P2:** mở rộng sau pilot.

Kích thước bubble biểu diễn severity; trục X biểu diễn coverage gap.

In [ ]:
gap_order = gaps.sort_values(['priority_rank', 'severity', 'coverage_gap'], ascending=[True, False, False])
fig2 = px.scatter(
    gap_order,
    x='coverage_gap', y='category', size='severity', color='priority',
    hover_name='gap', hover_data={'gap_id': True, 'closure_evidence': True, 'coverage_gap': ':.1f'},
    color_discrete_map={'P0': '#D62728', 'P1': '#F2A541', 'P2': '#4C78A8'},
    category_orders={'priority': ['P0', 'P1', 'P2']},
    title='Where the highest-risk data gaps are concentrated'
)
fig2.update_layout(height=620, xaxis_title='Industrial requirement − Cosmos coverage', yaxis_title='')
fig2.show(config=PLOTLY_CONFIG)
fig2.write_html(OUT / '02_priority_bubbles.html', include_plotlyjs='cdn')

priority_summary = (
    gaps.groupby('priority')
    .agg(gaps=('gap_id', 'count'), mean_severity=('severity', 'mean'), mean_coverage_gap=('coverage_gap', 'mean'))
    .reindex(['P0', 'P1', 'P2'])
    .round(2)
)
display(priority_summary)

In [ ]:
heat = gaps.pivot_table(index='category', columns='priority', values='coverage_gap', aggfunc='mean')
heat = heat.reindex(columns=['P0', 'P1', 'P2'])
plt.figure(figsize=(8, 8))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='YlOrRd', vmin=0, vmax=5, linewidths=.5, cbar_kws={'label': 'Mean coverage gap'})
plt.title('Gap heatmap by capability family and priority')
plt.xlabel('Priority')
plt.ylabel('Capability family')
plt.tight_layout()
plt.savefig(OUT / '03_gap_heatmap.png', dpi=180, bbox_inches='tight')
plt.show()

## 3. Data architecture: cái gì giữ lại, cái gì phải bổ sung?

Luồng bên trái là các nhóm dữ liệu Cosmos 3. Luồng bên phải cho thấy chúng chuyển thành capability nào, sau đó cần kết hợp với dữ liệu factory-native nào.

In [ ]:
labels = [
    'Cosmos Reasoner', 'SDG-Warehouse', 'SDG-PhyxSim', 'RobotSim / DROID', 'Cosmos audio',
    'Visual & spatial prior', 'Physical dynamics prior', 'Action prior', 'Temporal reasoning',
    'Real factory video', 'PLC / SCADA', 'Audio / vibration', 'Commands / interventions',
    'Quality / downtime', 'Maintenance / RUL', 'Industrial world-model episodes'
]
idx = {name: i for i, name in enumerate(labels)}
edges = [
    ('Cosmos Reasoner', 'Visual & spatial prior', 5),
    ('Cosmos Reasoner', 'Temporal reasoning', 4),
    ('SDG-Warehouse', 'Visual & spatial prior', 4),
    ('SDG-Warehouse', 'Physical dynamics prior', 2),
    ('SDG-PhyxSim', 'Physical dynamics prior', 5),
    ('RobotSim / DROID', 'Action prior', 5),
    ('RobotSim / DROID', 'Temporal reasoning', 2),
    ('Cosmos audio', 'Temporal reasoning', 2),
    ('Visual & spatial prior', 'Industrial world-model episodes', 5),
    ('Physical dynamics prior', 'Industrial world-model episodes', 5),
    ('Action prior', 'Industrial world-model episodes', 5),
    ('Temporal reasoning', 'Industrial world-model episodes', 4),
    ('Real factory video', 'Industrial world-model episodes', 5),
    ('PLC / SCADA', 'Industrial world-model episodes', 5),
    ('Audio / vibration', 'Industrial world-model episodes', 4),
    ('Commands / interventions', 'Industrial world-model episodes', 5),
    ('Quality / downtime', 'Industrial world-model episodes', 5),
    ('Maintenance / RUL', 'Industrial world-model episodes', 5),
]
sankey = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(label=labels, pad=15, thickness=18),
    link=dict(
        source=[idx[a] for a, b, v in edges],
        target=[idx[b] for a, b, v in edges],
        value=[v for a, b, v in edges]
    )
))
sankey.update_layout(title='Cosmos foundation + factory-native data → Industrial world-model episodes', height=620)
sankey.show(config=PLOTLY_CONFIG)
sankey.write_html(OUT / '04_data_architecture_sankey.html', include_plotlyjs='cdn')

## 4. Temporal-scale gap

Cosmos-style video mạnh ở frame → seconds → minutes. Factory world model còn phải nối được operation, batch, shift, machine life và maintenance event.

In [ ]:
temporal = pd.DataFrame([
    ('Control loop / vibration', 0.001, 0.1, 'Factory-native'),
    ('Contact / collision', 0.03, 10, 'Cosmos strong'),
    ('Robot action chunk', 0.2, 30, 'Cosmos strong'),
    ('Production operation', 10, 3600, 'Factory-native'),
    ('Batch / shift', 1800, 43200, 'Factory-native'),
    ('Degradation / maintenance', 86400, 15552000, 'Factory-native'),
], columns=['horizon', 'start_s', 'end_s', 'coverage'])

colors = {'Cosmos strong': '#4C78A8', 'Factory-native': '#F58518'}
fig3 = go.Figure()
for _, r in temporal.iterrows():
    fig3.add_trace(go.Scatter(
        x=[r.start_s, r.end_s], y=[r.horizon, r.horizon], mode='lines+markers',
        line=dict(width=10, color=colors[r.coverage]),
        marker=dict(size=8, color=colors[r.coverage]),
        name=r.coverage, legendgroup=r.coverage,
        showlegend=r.coverage not in [t.name for t in fig3.data],
        hovertemplate=f"{r.horizon}<br>{r.start_s:g}s → {r.end_s:g}s<extra></extra>"
    ))
fig3.update_layout(
    title='Temporal hierarchy required by a factory world model',
    xaxis=dict(type='log', title='Physical horizon in seconds (log scale)'),
    yaxis_title='', height=480, legend=dict(orientation='h', y=1.12)
)
fig3.show(config=PLOTLY_CONFIG)
fig3.write_html(OUT / '05_temporal_scale.html', include_plotlyjs='cdn')

## 5. Reuse, adapt hay collect?

Bảng này chuyển gap analysis thành quyết định dữ liệu. **Collect** không có nghĩa phải thu thập tất cả ngay; scope cụ thể được chốt ở task tiếp theo.

In [ ]:
decision_table = pd.DataFrame([
    ('General VLM/OCR/grounding', 'Reuse', 'Cosmos Reasoner foundation'),
    ('Depth, boxes, segmentation, camera geometry', 'Adapt', 'Synthetic pretraining + real calibration'),
    ('Rigid-body interactions and collision priors', 'Adapt', 'PhyxSim + real validation'),
    ('Robot manipulation action representation', 'Adapt', 'Map robot actions to target embodiment'),
    ('Real factory appearance and normal operation', 'Collect', 'Target machines, products, shifts and sites'),
    ('PLC/SCADA and process telemetry', 'Collect', 'Historian and edge sensor streams'),
    ('Factory commands and interventions', 'Collect', 'PLC/operator/robot command logs'),
    ('Quality, scrap, rework and downtime outcomes', 'Collect', 'MES/QMS/maintenance linkage'),
    ('Audio and vibration under load', 'Collect', 'Raw waveform + RPM/load/machine identity'),
    ('Long-horizon degradation and RUL', 'Collect', 'Lifecycle and maintenance histories'),
    ('Rare dangerous events', 'Generate after admission', 'Real anchor → simulation → A/B admission'),
], columns=['data_family', 'decision', 'implementation'])

display(decision_table.style.hide(axis='index').set_properties(**{'text-align': 'left'}))
decision_table.to_csv(OUT / 'reuse_adapt_collect.csv', index=False)

## 6. Ten representative factory episodes

Mỗi episode tuân theo data contract Cosmos-style:

`observation + physical_state + action/intervention → future_state + outcome`

In [ ]:
episodes = pd.DataFrame([
    ('Surface crack inspection', 'RGB', 'Divert part', 'Defect mask + reject outcome', 'MVTec AD + Cosmos grounding'),
    ('3D connector deformation', 'RGB + depth', 'Hold for review', 'Metric deviation + disposition', 'MVTec 3D-AD + Warehouse RGB-D'),
    ('Pump cavitation', 'Video + audio + sensors', 'Reduce speed', 'Vibration response', 'MIMII + Cosmos audio'),
    ('CNC spindle bearing fault', 'Video + audio + vibration', 'Feed hold', 'Maintenance confirmation', 'CWRU + PhyxSim'),
    ('Robot insertion misalignment', 'Multi-view video + force', 'Retract and realign', 'Insertion success', 'IndustReal + RobotSim/DROID'),
    ('Assembly step omission', 'Overhead + egocentric video', 'Pause and prompt', 'Recovered sequence', 'Assembly101 + temporal reasoning'),
    ('Conveyor package jam', 'RGB-D + motor current', 'Slow and divert', 'Jam avoided', 'PhyxSim forward dynamics'),
    ('Forklift-worker near miss', 'Multi-view RGB-D', 'Emergency brake', 'Collision avoided', 'SDG-Warehouse'),
    ('Reactor cooling fault', 'Thermal + process sensors', 'Switch valve', 'Temperature recovery', 'Tennessee Eastman + action tokens'),
    ('Compressor degradation', 'Video + audio + telemetry', 'Shift load + maintenance', 'RUL / alarm horizon', 'NASA PCoE + future prediction'),
], columns=['scenario', 'observation', 'action', 'outcome', 'research_basis'])
display(episodes.style.hide(axis='index'))
episodes.to_csv(OUT / 'representative_factory_episodes.csv', index=False)

## 7. Automatic conclusions and next-task inputs

In [ ]:
top_p0 = gaps[gaps.priority == 'P0'].sort_values(['severity', 'coverage_gap'], ascending=False).head(6)
strong = cap[cap.cosmos_coverage >= 4.0].sort_values('cosmos_coverage', ascending=False)
collect = cap[cap.decision == 'Collect'].sort_values('gap', ascending=False)

display(Markdown('### Reuse from Cosmos 3'))
for x in strong.capability:
    print('•', x)

display(Markdown('### Highest-priority factory-native data'))
for _, r in top_p0.iterrows():
    print(f"• {r.gap} — closure: {r.closure_evidence}")

display(Markdown('### Decisions required before the next task can close'))
next_inputs = [
    'First deployment use case',
    'Target factory/site and machine classes',
    'Prediction horizon and operational outcome',
    'Available cameras, sensors and command logs',
    'Safety-critical error tolerance',
    'Privacy, security and retention boundary',
]
for x in next_inputs:
    print('•', x)

summary = {
    'gap_count': int(len(gaps)),
    'P0': int((gaps.priority == 'P0').sum()),
    'P1': int((gaps.priority == 'P1').sum()),
    'P2': int((gaps.priority == 'P2').sum()),
    'mean_coverage_gap': round(float(gaps.coverage_gap.mean()), 3),
    'top_p0_gaps': top_p0.gap.tolist(),
    'reusable_capabilities': strong.capability.tolist(),
    'factory_native_collection': collect.capability.tolist(),
}
(OUT / 'analysis_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

## 8. Primary references

- [NVIDIA Cosmos 3 technical report](https://research.nvidia.com/labs/cosmos-lab/cosmos3/technical-report.pdf)
- [NVIDIA SDG-Warehouse](https://huggingface.co/datasets/nvidia/PhysicalAI-WorldModel-Synthetic-Warehouse-Operations-Scenes)
- [NVIDIA SDG-PhyxSim](https://huggingface.co/datasets/nvidia/PhysicalAI-WorldModel-Synthetic-Physical-Interaction-Scenes)
- [NVIDIA SDG-RobotSim](https://huggingface.co/datasets/nvidia/PhysicalAI-WorldModel-Synthetic-Embodied-Robot-Scenes)
- [MVTec AD](https://www.mvtec.com/company/research/datasets/mvtec-ad)
- [MVTec 3D-AD](https://www.mvtec.com/research-teaching/datasets/mvtec-3d-ad)
- [MIMII](https://zenodo.org/records/3384388)
- [CWRU Bearing Data Center](https://engineering.case.edu/bearingdatacenter/download-data-file)
- [Assembly101](https://assembly-101.github.io/)
- [NVIDIA IndustReal](https://developer.nvidia.com/blog/transferring-industrial-robot-assembly-tasks-from-simulation-to-reality/)
- [NASA Prognostics Center of Excellence](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/)

In [ ]:
gaps.to_csv(OUT / 'cosmos3_industrial_gap_matrix.csv', index=False)
cap.to_csv(OUT / 'capability_coverage.csv', index=False)

archive = Path('/content/cosmos3_industrial_gap_analysis_artifacts.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUT.rglob('*')):
        if path.is_file():
            zf.write(path, path.relative_to(OUT.parent))

print('Created:', archive)
print('Size:', round(archive.stat().st_size / 1024**2, 2), 'MiB')
print('Files:')
for path in sorted(OUT.iterdir()):
    print(' -', path.name)